<a href="https://colab.research.google.com/github/GUNAPILLCO/neural_profit/blob/main/3_dise%C3%B1o_entrenamiento_evaluacion/3_2_Generacion_X_y.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 3.2. Generación de X e y para modelos

## 0. Clonado de Repositorio, instalación de librería e importación.

In [1]:
#Clonamos el repo
#LINK DE REPOSITORIO: https://github.com/GUNAPILLCO/neural_profit

!git clone https://github.com/GUNAPILLCO/neural_profit.git

Cloning into 'neural_profit'...
remote: Enumerating objects: 350, done.
remote: Counting objects: 100% (71/71), done.
remote: Compressing objects: 100% (65/65), done.
remote: Total 350 (delta 43), reused 11 (delta 6), pack-reused 279 (from 2)
Receiving objects: 100% (350/350), 246.53 MiB | 15.20 MiB/s, done.
Resolving deltas: 100% (193/193), done.
Updating files: 100% (57/57), done.


In [4]:
# !{sys.executable} -m pip install -q pandas_market_calendars  # Solo si usás horarios de mercados
!{sys.executable} -m pip install -q ta
print("Librerías instaladas: ta")

  Preparing metadata (setup.py) ... done
Librerías instaladas: ta


In [5]:
import sys
import warnings
warnings.filterwarnings('ignore')

# Utilidades del sistema y fechas
import os
import glob
import requests
from datetime import datetime, timedelta

# Procesamiento de datos
import pandas as pd
import numpy as np

# Visualización
import matplotlib.pyplot as plt
import seaborn as sns
from tabulate import tabulate

# Análisis técnico
import ta
from ta.momentum import StochasticOscillator, ROCIndicator
from ta.volatility import BollingerBands, AverageTrueRange

# Estadística
from scipy.stats import spearmanr

#que es?
#from tqdm.notebook import tqdm
from tqdm import tqdm
# Modelos ML
#from xgboost import XGBRegressor
#from sklearn.metrics import mean_squared_error, r2_score

# Calendario de mercados (descomentar si lo necesitás)
# import pandas_market_calendars as mcal

## 1. Carga de datasets train, valid y test.

In [6]:
def load_df():
    """
    Función para cargar un archivo Parquet desde el repositorio clonado
    """
    # Definir la URL del archivo Parquet en GitHub
    df_path_mnq = '/content/neural_profit/3_diseño_entrenamiento_evaluacion/mnq_model.parquet'
    df_path_factores = '/content/neural_profit/3_diseño_entrenamiento_evaluacion/df_factores.parquet'
    df_path_train = '/content/neural_profit/3_diseño_entrenamiento_evaluacion/mnq_train.parquet'
    df_path_valid = '/content/neural_profit/3_diseño_entrenamiento_evaluacion/mnq_valid.parquet'
    df_path_test = '/content/neural_profit/3_diseño_entrenamiento_evaluacion/mnq_test.parquet'

    # Leer el archivo Parquet y cargarlo en un DataFrame
    df_model = pd.read_parquet(df_path_mnq)
    df_factores = pd.read_parquet (df_path_factores)
    df_train = pd.read_parquet(df_path_train)
    df_valid = pd.read_parquet(df_path_valid)
    df_test = pd.read_parquet(df_path_test)

    return df_model, df_factores, df_train, df_valid, df_test

In [7]:
mnq_model, indicadores_tecnicos, mnq_train, mnq_valid, mnq_test = load_df()

## 1.1. Información de los datasets

In [8]:
def info_dataset (df): # Contar valores únicos en la columna 'date'
  num_dias = df['date'].nunique()
  print(f"Cantidad de días: {num_dias}")

  # Filtrar valores válidos
  validos_por_dia = df.dropna(subset=['target_return_30']).groupby('date').size()

  # Calcular el promedio
  promedio_por_fecha = validos_por_dia.mean()
  print(f"Valores por día: {int(promedio_por_fecha)}")

  primer_hora = df.index[0].strftime('%H:%M')
  ultima_hora = df.index[-1].strftime('%H:%M')
  zona_horaria = df.index[0].tzinfo
  print(f"Hora diaria de inicio {primer_hora}")
  print(f"Hora diaria de final {ultima_hora}")
  print(f"Zona horaria: {zona_horaria}")

In [9]:
info_dataset(mnq_train)

Cantidad de días: 917
Valores por día: 361
Hora diaria de inicio 09:30
Hora diaria de final 15:30
Zona horaria: America/New_York


In [10]:
info_dataset(mnq_valid)

Cantidad de días: 197
Valores por día: 361
Hora diaria de inicio 09:30
Hora diaria de final 15:30
Zona horaria: America/New_York


In [11]:
info_dataset(mnq_test)

Cantidad de días: 197
Valores por día: 361
Hora diaria de inicio 09:30
Hora diaria de final 15:30
Zona horaria: America/New_York


## 2. Generación de ventanas X e y

In [12]:
target_column = "target_return_30"
features = mnq_model.columns.tolist()
features.remove(target_column)
features.remove('date')

window_size = 60

In [14]:
#Función para generar ventanas y vectorizarlas
def generar_ventanas(df, features, target_col, window_size):
    X, y = [], []
    for fecha, grupo in tqdm(df.groupby("date"), desc="Procesando días"):
        grupo = grupo.reset_index(drop=True)
        for i in range(len(grupo) - window_size):
            ventana = grupo.loc[i:i+window_size-1, features]
            if ventana.isnull().any().any():
                continue
            vector = ventana.values.flatten()
            target = grupo.loc[i+window_size-1, target_col]
            X.append(vector)
            y.append(target)
    return np.array(X), np.array(y)

In [15]:
print('Generando X_train e y_train: \n')
X_train, y_train = generar_ventanas(mnq_train, features, target_column, window_size)

print('Generando X_valid e y_valid: \n')
X_valid, y_valid = generar_ventanas(mnq_valid, features, target_column, window_size)

print('Generando X_test e y_test: \n')
X_test, y_test = generar_ventanas(mnq_test, features, target_column, window_size)

Generando X_train e y_train: 



Procesando días: 100%|██████████| 917/917 [04:31<00:00,  3.38it/s]


In [33]:
print(f"X_train info: {X_train.shape[0]} ventanas aplanadas en {X_train.shape[1]} números, el equivalente a la window size ({window_size}) por la cantidad de features ({len(features)})")
print(f"y_train info: {y_train.shape[0]} valores que corresponden al retorno a 30 minutos ({target_column})")

X_train info: 276017 ventanas aplanadas en 1260 números, el equivalente a la window size (60) por la cantidad de features (21)
y_train info: 276017 valores que corresponden al retorno a 30 minutos (target_return_30)
